# Missed Trash Pickups: Red River Waste Solutions
**Juan Zavala** · AI for Analytics · hubNashville service requests, Nov 2017 to Nov 2019

Claude worked as a **pair programmer** here, not an answer generator. Every step follows the same pattern: Claude explains its approach in plain English, I review that plan, and only then is code written and run.

| Label | Meaning |
|---|---|
| **[ME]** | Code I wrote myself |
| **[CLAUDE]** | Code Claude wrote after I approved its plan |
| **[REVIEW]** | What I changed or questioned in Claude's plan or code |
| **[CHECK]** | A validation step: does the code do what we think it does? |

Part 1 (planning, no code) is in [`part1_planning.md`](part1_planning.md).

In [1]:
# [ME] Load the data
import pandas as pd

pd.set_option('display.max_colwidth', 110)
pd.set_option('display.width', 200)

trash = pd.read_csv(
    'data/trash_hauler_report_with_lat_lng.csv',
    dtype={'Request Number': str, 'Zip Code': str, 'Trash Route': str, 'Council District': str},
)
trash['Date Opened'] = pd.to_datetime(trash['Date Opened'], format='%m/%d/%Y')

print(trash.shape)
trash.head(3)

(20226, 13)


,Request Number,Date Opened,Request,Description,Incident Address,Zip Code,Trash Hauler,Trash Route,Council District,State Plan X,State Plan Y,LONGITUDE,LATITUDE
0,25270,2017-11-01,Trash - Backdoor,"house with the wheel chair ramp, they share driveway, in back driveway. 3817 Crouch Dr near NES light post...",3817 Crouch Dr,37207,RED RIVER,3205,2,1727970.412,686779.4781,-86.815392,36.217292
1,25274,2017-11-01,Trash - Curbside/Alley Missed Pickup,Curb/Trash miss Tuesday.,4028 Clarksville Pike,37218,RED RIVER,4202,1,1721259.366,685444.7996,-86.838103,36.213470
2,25276,2017-11-01,Trash - Curbside/Alley Missed Pickup,Curb/trash miss Tuesday.,6528 Thunderbird Dr,37209,RED RIVER,4205,20,1707026.753,659887.4716,-86.885562,36.142923


In [2]:
# [ME] First look at the columns that matter
print(trash['Request'].value_counts(), '\n')
print(trash['Trash Hauler'].value_counts(dropna=False), '\n')
print('Dates:', trash['Date Opened'].min().date(), 'to', trash['Date Opened'].max().date())
print('Distinct raw addresses:', trash['Incident Address'].nunique())

Request
Trash - Curbside/Alley Missed Pickup    15028
Trash - Backdoor                         2629
Trash Collection Complaint               2312
Damage to Property                        257
Name: count, dtype: int64 

Trash Hauler
RED RIVER    14395
METRO         3512
WASTE IND     1350
NaN            901
Metro           68
Name: count, dtype: int64 

Dates: 2017-11-01 to 2019-11-01
Distinct raw addresses: 14121


---
# Part 2.1: Clean and standardize the address column

### Prompt I gave Claude
> "I'm working in Python with pandas. Help me clean and standardize the `Incident Address` column so the same house is always written the same way. **Explain your approach in plain English first. Don't write code until I approve it.** Use built-in pandas methods instead of a custom function where you can."

### Claude's approach (before any code)
1. **Upper-case everything**, so "Dr" and "DR" match.
2. **Keep only the street part.** Many addresses end in `, NASHVILLE, TN 37205, UNITED STATES` and many don't, so cut everything after the first comma.
3. **Remove punctuation noise.** Drop periods (`RD.` becomes `RD`) and turn `#` into a space (`#416` becomes `416`).
4. **Collapse repeated spaces** and trim the ends.
5. **Abbreviate street suffixes** to the USPS short form: STREET to ST, AVENUE to AVE, DRIVE to DR, and so on.
6. **Abbreviate directions** like NORTH to N and WEST to W.
7. **Leave unit numbers alone**, so `1007 ALICE ST B` stays a different premises from `1007 ALICE ST`.

### [REVIEW] What I changed before running it
I tested Claude's plan against real rows before accepting it, and found three problems:

| Claude's plan said | Problem I found in the data | What I changed |
|---|---|---|
| Step 6: change WEST to W everywhere | **West End Ave** is a real street name (`3516 WEST END AVE`), and so are `WEST BEND DR`, `EAST LN` and `SOUTH ST`. Changing them everywhere would rewrite the street name itself. | Only change a direction when it's the **last word**, like `214 19TH AVENUE NORTH` becoming `214 19TH AVE N`. |
| Step 2: cutting at the comma removes the city and ZIP | Some rows have the city and ZIP with **no comma**: `2304 EASTLAND AVE NASHVILLE TN 37206` and `1916 CASTLEMAN DRIVE 37215`. | Also remove a trailing `[city] [TN] 12345` pattern. I made sure it doesn't touch `ANTIOCH PIKE` or `TENNESSEE AVE`, which only match when a 5-digit ZIP follows. |
| No step for bad values | One address is literally `NO ADDRESS FOUND.`, and some have **no house number** (`XAVIER DR`) or are **intersections** (`TRIMBLE RD & LINDAWOOD DR`). | Turn placeholders into missing values, and add a `has_house_number` flag so we can leave out rows that can't be tied to one premises. |

In [3]:
# [CLAUDE] Address cleaning, with my [REVIEW] edits included. Every step is a built-in pandas .str method.
SUFFIXES = {
    'STREET': 'ST', 'AVENUE': 'AVE', 'DRIVE': 'DR', 'ROAD': 'RD', 'LANE': 'LN',
    'COURT': 'CT', 'CIRCLE': 'CIR', 'PLACE': 'PL', 'BOULEVARD': 'BLVD',
    'PARKWAY': 'PKWY', 'HIGHWAY': 'HWY', 'TERRACE': 'TER', 'TRAIL': 'TRL',
}
SUFFIX_PATTERN = r'\b(' + '|'.join(SUFFIXES) + r')\b'          # whole words only

# [REVIEW] city/state/ZIP written without a comma, e.g. "... AVE NASHVILLE TN 37206"
CITY_ZIP_PATTERN = (r'\s+(?:NASHVILLE|ANTIOCH|HERMITAGE|MADISON|OLD HICKORY|GOODLETTSVILLE|BRENTWOOD)?'
                    r'\s*(?:TN|TENNESSEE)?\s*\d{5}$')

trash['clean_address'] = (
    trash['Incident Address']
        .str.upper()
        .str.split(',').str[0]                                   # keep the street part only
        .str.replace('.', '', regex=False)                       # RD. -> RD
        .str.replace('#', ' ', regex=False)                      # #416 -> 416
        .str.replace(r'\s+', ' ', regex=True)                    # collapse spaces
        .str.strip()
        .str.replace(CITY_ZIP_PATTERN, '', regex=True)           # [REVIEW] city/ZIP without a comma
        .str.replace(SUFFIX_PATTERN, lambda m: SUFFIXES[m.group(0)], regex=True)
        .str.replace(r' (NORTH|SOUTH|EAST|WEST)$',               # [REVIEW] last word only
                     lambda m: ' ' + m.group(1)[0], regex=True)
        .str.strip()
        .replace({'': pd.NA, 'NO ADDRESS FOUND': pd.NA})         # [REVIEW] placeholders -> missing
)

# [REVIEW] a premises needs a house number; intersections and bare street names don't qualify
trash['has_house_number'] = trash['clean_address'].str.match(r'^\d', na=False)

print('Distinct addresses, raw:        ', trash['Incident Address'].nunique())
print('Distinct addresses, upper-case: ', trash['Incident Address'].str.upper().nunique())
print('Distinct addresses, cleaned:    ', trash['clean_address'].nunique())
print('Rows without a usable address:  ', (~trash['has_house_number']).sum())

Distinct addresses, raw:         14121
Distinct addresses, upper-case:  13781
Distinct addresses, cleaned:     12349
Rows without a usable address:   69


In [4]:
# [CHECK] Show raw spellings that now share one clean address
merged = (trash.dropna(subset=['clean_address'])
               .groupby('clean_address')['Incident Address']
               .agg(['nunique', 'unique']))
print('Clean addresses that combined 2+ raw spellings:', (merged['nunique'] > 1).sum())

examples = ['1007 ALICE ST', '101 AILEEN CT', '3847 CROSS CREEK RD', '214 19TH AVE N',
            '2304 EASTLAND AVE', '1916 CASTLEMAN DR', '4049 MOSS RD']
merged.loc[merged.index.intersection(examples), 'unique'].to_frame('raw spellings combined')

Clean addresses that combined 2+ raw spellings: 1493


,raw spellings combined
clean_address,
1007 ALICE ST,"[1007 Alice St, Nashville, Tennessee, 37218, 1007 Alice St, Nashville, TN 37218, United States, 1007 ALICE..."
101 AILEEN CT,"[101 Aileen Ct, Nashville, TN 37013, United States, 101 Aileen Ct, Antioch, TN 37013, USA]"
1916 CASTLEMAN DR,"[1916 Castleman Dr, 1916 Castleman Drive 37215, 1916 castleman dr]"
214 19TH AVE N,"[214 19th Ave N, 214 19th Avenue North, Nashville, TN, USA, 214 19th Ave N, Nashville, TN 37203, United St..."
2304 EASTLAND AVE,"[2304 Eastland Ave, 2304 Eastland Ave Nashville Tn 37206, 2304 Eastland Ave, Nashville, TN 37206, United S..."
3847 CROSS CREEK RD,"[3847 Cross Creek Rd, 3847 Cross Creek Road, Nashville, TN, USA, 3847 Cross Creek Rd, Nashville, TN 37215,..."
4049 MOSS RD,[4049 Moss Rd.]


In [5]:
# [CHECK] Did anything slip through? Nothing should still contain a city, state, country or comma.
leftover = trash['clean_address'].str.contains(r'NASHVILLE|UNITED STATES|\bUSA\b|,|\bTN \d|\d{5}$', na=False)
print('Suspicious leftovers:', trash.loc[leftover, 'clean_address'].unique())

# [CHECK] West End Ave must survive unchanged (the [REVIEW] fix for step 6)
print(trash.loc[trash['clean_address'].str.contains('WEST END', na=False), 'clean_address'].unique()[:5])

Suspicious leftovers: []
['3516 WEST END AVE' '112 WEST END PL' '102 WEST END CLOSE'
 '3631 WEST END AVE' '109 WEST END CLOSE']


**Result:** No clean address still contains a city, state, country, ZIP or comma, and West End Ave came through unchanged.

**[REVIEW]** My first version of this check flagged `3712 LAUSANNE DR` and three of its neighbors, because "LAUSANNE" contains the letters U-S-A. The check was wrong, not the cleaning, so I made it match `USA` only as a whole word. Validation code can have bugs too.

In [6]:
# [CHECK] Units: is "1007 ALICE ST B" ever written as "1007B ALICE ST"? If so, those would stay split.
letter_first = trash['clean_address'].str.extract(r'^(\d+)([A-Z]) (.+)$').dropna()
letter_first_as_unit = letter_first[0] + ' ' + letter_first[2] + ' ' + letter_first[1]
both_forms = letter_first_as_unit.isin(set(trash['clean_address'].dropna()))
print('Addresses written like 3804A TULANE CT:', len(letter_first.drop_duplicates()))
print('...that also appear as 3804 TULANE CT A:', both_forms.sum())

Addresses written like 3804A TULANE CT: 53
...that also appear as 3804 TULANE CT A: 9


In [7]:
# [REVIEW] Fix it: move a letter stuck to the house number to the end, as a unit (3804A TULANE CT -> 3804 TULANE CT A)
trash['clean_address'] = trash['clean_address'].str.replace(r'^(\d+)([A-Z]) (.+)$', r'\1 \3 \2', regex=True)
trash['has_house_number'] = trash['clean_address'].str.match(r'^\d', na=False)

print('Letter-stuck house numbers left:', trash['clean_address'].str.match(r'^\d+[A-Z] ', na=False).sum())
print('Distinct addresses, cleaned (final):', trash['clean_address'].nunique())
trash.loc[trash['Incident Address'].str.upper().str.startswith('3804A TULANE', na=False),
          ['Incident Address', 'clean_address']].head(2)

Letter-stuck house numbers left: 0
Distinct addresses, cleaned (final): 12341


,Incident Address,clean_address
6079,"3804A Tulane Ct, Nashville, TN 37215, United States",3804 TULANE CT A


**Result:** 53 addresses had the unit letter stuck to the house number (`3804A TULANE CT`). For 9 of them, the same unit also appears in the normal form (`3804 TULANE CT A`), so one apartment was being counted as two addresses. After the fix, none are left. Overall, **14,121 raw spellings became 12,341 standardized addresses.**

### [REVIEW] Is the pandas version actually faster than a custom function?
The assignment says pandas methods are often more efficient than a custom function. Instead of taking that on trust, I timed the same cleaning three ways on all 20,226 rows.

In [8]:
# [CHECK] Time three ways of doing the same cleaning
import re, time

def clean_one(addr):                      # custom function, written carefully
    if pd.isna(addr):
        return addr
    addr = addr.upper().split(',')[0].replace('.', '').replace('#', ' ')
    addr = re.sub(r'\s+', ' ', addr).strip()
    addr = re.sub(CITY_ZIP_PATTERN, '', addr)
    words = [SUFFIXES.get(w, w) for w in addr.split()]
    if words and words[-1] in ('NORTH', 'SOUTH', 'EAST', 'WEST'):
        words[-1] = words[-1][0]
    return ' '.join(words)

def clean_one_naive(addr):                # custom function, the way it's often first written
    if pd.isna(addr):
        return addr
    addr = addr.upper().split(',')[0].replace('.', '').replace('#', ' ')
    addr = re.sub(r'\s+', ' ', addr).strip()
    addr = re.sub(CITY_ZIP_PATTERN, '', addr)
    for long, short in SUFFIXES.items():  # 13 separate regex passes per row
        addr = re.sub(r'\b' + long + r'\b', short, addr)
    return re.sub(r' (NORTH|SOUTH|EAST|WEST)$', lambda m: ' ' + m.group(1)[0], addr)

def vectorized(s):
    return (s.str.upper().str.split(',').str[0]
             .str.replace('.', '', regex=False).str.replace('#', ' ', regex=False)
             .str.replace(r'\s+', ' ', regex=True).str.strip()
             .str.replace(CITY_ZIP_PATTERN, '', regex=True)
             .str.replace(SUFFIX_PATTERN, lambda m: SUFFIXES[m.group(0)], regex=True)
             .str.replace(r' (NORTH|SOUTH|EAST|WEST)$', lambda m: ' ' + m.group(1)[0], regex=True))

def best_of_5(fn):
    times = []
    for _ in range(5):
        start = time.perf_counter(); result = fn(); times.append(time.perf_counter() - start)
    return min(times) * 1000, result

addr = trash['Incident Address']
results = {name: best_of_5(fn) for name, fn in [
    ('pandas .str methods', lambda: vectorized(addr)),
    ('.apply(custom function)', lambda: addr.apply(clean_one)),
    ('.apply(naive custom function)', lambda: addr.apply(clean_one_naive)),
]}
outputs = [r[1].fillna('<NA>').astype(str) for r in results.values()]
print('All three give identical results:', all((o == outputs[0]).all() for o in outputs))
pd.Series({k: round(v[0], 1) for k, v in results.items()}, name='milliseconds (best of 5)')

All three give identical results: True


pandas .str methods               83.3
.apply(custom function)           70.9
.apply(naive custom function)    271.5
Name: milliseconds (best of 5), dtype: float64

**What I found:** The "pandas is faster" advice is only half true for this step.

* A **carefully written custom function** is about as fast as the pandas `.str` chain. Both take well under a tenth of a second for 20,226 rows. pandas string methods still visit each value one at a time behind the scenes, so they don't get the big speedup that math on numbers does.
* A **naive custom function**, the kind that runs 13 separate regex replacements on every row, is about **3 to 4 times slower**.
* All three give **identical results**.

Your timings will differ a little on another computer, but the pattern holds. **Takeaway:** for string cleaning, the pandas version wins on readability more than speed. The real speed difference shows up in Parts 3 and 4, where `groupby` with `cumcount` and rolling windows replace looping over 11,000 addresses one by one.

---
# Part 2.2: Identify rows that represent missed pickups

### Prompt I gave Claude
> "Now help me decide which rows are missed pickups. Explain your rules in plain English first, with no code yet. Then help me hand-check a small sample."

### Claude's approach (before any code)
1. **By request type.** `Trash - Curbside/Alley Missed Pickup` and `Trash - Backdoor` are missed pickups. `Damage to Property` is not. `Trash Collection Complaint` is a judgment call, so check a sample before deciding.
2. **Must have a usable address**, with a house number. Otherwise it can't count toward repeats at "the same premises."
3. **One missed pickup per address per day.** If the same address is reported twice on the same day, it's one missed pickup reported twice, not two.
4. **Clean up the hauler column.** `Metro` and `METRO` are the same value. Fines only apply to **Red River**, so we need to know each row's hauler.

### [REVIEW] My notes on the plan
* Rule 4: 901 rows have **no hauler at all**. Part 1 suggested filling them from `Trash Route`, but I checked and **those rows have no route either** (only 32 of the 716 blank-hauler missed pickups have one). Instead, I fill the hauler from **other requests at the same clean address**, and only when that address has exactly one hauler on record.
* Rule 3: I wanted to see real examples before accepting this, so the manual check below includes an address reported 4 times on one day.

In [9]:
# [CLAUDE] Rule 1: request types that are missed pickups
MISSED_TYPES = ['Trash - Curbside/Alley Missed Pickup', 'Trash - Backdoor']
trash['is_missed_type'] = trash['Request'].isin(MISSED_TYPES)

# [CLAUDE] Rule 4: one spelling per hauler
trash['hauler'] = trash['Trash Hauler'].str.upper()

# [REVIEW] Fill a blank hauler from other requests at the same address, only if they all agree
known = trash.dropna(subset=['hauler', 'clean_address']).groupby('clean_address')['hauler']
single_hauler = known.first()[known.nunique() == 1]
was_blank = trash['hauler'].isna()
trash['hauler'] = trash['hauler'].fillna(trash['clean_address'].map(single_hauler))
print('Blank haulers before:', was_blank.sum(),
      '| filled from same address:', (was_blank & trash['hauler'].notna()).sum(),
      '| still blank:', trash['hauler'].isna().sum())
trash['hauler'].value_counts(dropna=False)

Blank haulers before: 901 | filled from same address: 542 | still blank: 359


hauler
RED RIVER    14810
METRO         3679
WASTE IND     1378
NaN            359
Name: count, dtype: int64

In [10]:
# [CLAUDE] Rules 2 and 3: usable address, then one missed pickup per address per day
candidates = trash[trash['is_missed_type'] & trash['has_house_number']].copy()
candidates = candidates.sort_values(['clean_address', 'Date Opened', 'Request Number'])
candidates['same_day_repeat'] = candidates.duplicated(['clean_address', 'Date Opened'])

missed = candidates[~candidates['same_day_repeat']].copy()

# [ME] How many rows each rule removes, step by step
funnel = pd.Series({
    'All service requests':                         len(trash),
    'Missed-pickup request types':                  trash['is_missed_type'].sum(),
    '...with a usable address (house number)':      len(candidates),
    '...one per address per day (missed pickups)':  len(missed),
    '...where the hauler is Red River':             (missed['hauler'] == 'RED RIVER').sum(),
}, name='rows')
funnel.to_frame()

,rows
All service requests,20226
Missed-pickup request types,17657
...with a usable address (house number),17609
...one per address per day (missed pickups),17243
...where the hauler is Red River,12946


---
## [CHECK] Manual review of a small subset
The point is to read actual rows with my own eyes, not just trust the counts.

**Check 1: Do 10 random rows labeled "missed pickup" actually describe a missed pickup?**

In [11]:
# [CHECK] 1: random sample of rows labeled as missed pickups (fixed seed so it's repeatable)
missed.sample(10, random_state=42)[['Request', 'Description', 'Incident Address', 'clean_address', 'hauler']]

,Request,Description,Incident Address,clean_address,hauler
13882,Trash - Curbside/Alley Missed Pickup,The entire side of the street was missed,1002 N Graycroft Ave,1002 N GRAYCROFT AVE,RED RIVER
699,Trash - Curbside/Alley Missed Pickup,Missed.,2805 Lera Jones Dr,2805 LERA JONES DR,RED RIVER
10443,Trash - Curbside/Alley Missed Pickup,1 cart,"409 Windsor Ct, Nashville, TN 37211, United States",409 WINDSOR CT,RED RIVER
11664,Trash - Curbside/Alley Missed Pickup,"One side of the street not service, Tuesday missed. Curb 3118 and others.","3118 Moorewood Dr, Nashville, TN 37207, United States",3118 MOOREWOOD DR,RED RIVER
18483,Trash - Curbside/Alley Missed Pickup,Entire street/miss trash/last Friday. Curb,"3653 Pin Hook Rd, Nashville, TN 37013, United States",3653 PIN HOOK RD,RED RIVER
9736,Trash - Curbside/Alley Missed Pickup,MISSED- MISSED THE ENTIRE CUL DE SAC,"3004 COUNTRY MEADOW RD, 37013",3004 COUNTRY MEADOW RD,RED RIVER
11104,Trash - Curbside/Alley Missed Pickup,Missed pickup,"941 Havenhill Dr, Nashville, TN 37217, United States",941 HAVENHILL DR,RED RIVER
12414,Trash - Curbside/Alley Missed Pickup,Missed trash,"576 Whispering Hills Dr, Nashville, TN 37211, United States",576 WHISPERING HILLS DR,RED RIVER
12594,Trash - Curbside/Alley Missed Pickup,Trash/curb missed Thursday.,"642 Devon Dr, Nashville, TN 37220, United States",642 DEVON DR,RED RIVER
8987,Trash - Curbside/Alley Missed Pickup,Customer stated this is the 5th missed pickup. She stated there was a man washing carpets outside but ther...,"1820 Champions Dr, Nashville, TN 37211, United States",1820 CHAMPIONS DR,RED RIVER


**Result:** All 10 rows really are missed pickups, so the request type is a reliable label. Two things stood out:

* **4 of the 10 say a whole street, one side of a street or a cul-de-sac was missed**, but each request is filed under one address. The neighbors only count if they called too, so the real number of missed pickups is probably **higher** than the data shows. Code can't fix that. It's a limitation to report.
* A description like "1 cart" is too short to judge alone. That's why I rely on the request type and not on searching descriptions for keywords.

**Check 2: Should `Trash Collection Complaint` rows that mention "miss" be counted?**

In [12]:
# [CHECK] 2: complaints that mention "miss"
complaints = trash[(trash['Request'] == 'Trash Collection Complaint')
                   & trash['Description'].str.contains('miss', case=False, na=False)]
print('Complaint rows mentioning "miss":', len(complaints), 'of',
      (trash['Request'] == 'Trash Collection Complaint').sum())

# How many of them just repeat a missed-pickup request already filed for that address that day?
already_counted = complaints.set_index(['clean_address', 'Date Opened']).index.isin(
    candidates.set_index(['clean_address', 'Date Opened']).index)
print('...already covered by a missed-pickup request at the same address on the same day:', already_counted.sum())
complaints.sample(10, random_state=7)[['Date Opened', 'Description', 'clean_address']]

Complaint rows mentioning "miss": 691 of 2312
...already covered by a missed-pickup request at the same address on the same day: 36


,Date Opened,Description,clean_address
13140,2019-05-30,MISSED...ENTIRE COMPLEX OF APPX 42 HOMES...GLOSTER SQUARE. RESIDENT SAYS IT IS RIDICULOUS THAT EACH INDIVI...,2138 GOLF CLUB LN
9293,2019-02-01,Caller said she has been missed every friday for a month. They trash truck has already come this morning ...,3004 FIELDSTONE DR
5792,2018-08-29,Missed every other week customer tired of calling the Hub to make driver come back to get trash 3 days lat...,5105 BUENA VISTA PIKE
93,2017-11-03,missed - everyone except his picked up,641 TRUXTON CT
15298,2019-07-22,"Whole circle missed, & Longhunter Ct. Trash missed last Friday.",209 LONGHUNTER CIR
17226,2019-08-21,RESIDENT WANTS TO KNOW WHY HIS ADDRESS IS CONSISTENTLY MISSED. THIS IS HAPPENING FREQUENTLY.,1501 RIVERSIDE DR
20096,2019-10-30,today makes 3 weeks missed trash/curb {sit's on 40th Ave N} side. {Wednesday}\ndoes not have access to all...,4001 NEBRASKA AVE
564,2017-12-07,cust says pickup was missed for him 3 weeks in a row,5513 MORROW RD
18408,2019-09-20,"Has Backdoor/Trash pickup, 2 weeks of being missed, elderly and if set any closer of where she places her ...",4285 LAVERGNE COUCHVILLE PIKE
19547,2019-10-17,**MISSED AGAIN YESTERDAY****SUPERVISOR DID NOT MAKE CONTACT AS REQUESTED***PLEASE CALL THIS TIME********SI...,2813 DELAWARE AVE


**Decision: leave complaints out.** Many of these rows are about missed trash, but most describe a **pattern** ("missed every friday for a month", "3 weeks in a row", "consistently missed") rather than one missed pickup on the date the request was opened. Only 36 of the 691 fall on the same address and day as a missed-pickup request, and those would be duplicates anyway. The other 655 may point to extra misses, but there's no way to tell which pickup date each one means. Counting each as one missed pickup would be a guess, so the Part 3 totals are **conservative** (on the low side).

**Check 3: Does the same-day rule do the right thing?** Here is `4049 MOSS RD` with every raw row.

In [13]:
# [CHECK] 3: one address, every raw row, side by side with what the rules kept
cols = ['Request Number', 'Date Opened', 'Request', 'Incident Address', 'Description', 'hauler']
moss = candidates.loc[candidates['clean_address'] == '4049 MOSS RD', cols + ['same_day_repeat']]
moss

,Request Number,Date Opened,Request,Incident Address,Description,hauler,same_day_repeat
320,28848,2017-11-17,Trash - Curbside/Alley Missed Pickup,4049 Moss Rd.,Had trash at road last night but still missed pickup this morning,NaN,False
321,28849,2017-11-17,Trash - Curbside/Alley Missed Pickup,4049 Moss Rd.,Had trash at road last night but still missed pickup this morning,NaN,True
322,28850,2017-11-17,Trash - Curbside/Alley Missed Pickup,4049 Moss Rd.,Had trash at road last night but still missed pickup this morning,NaN,True
323,28851,2017-11-17,Trash - Curbside/Alley Missed Pickup,4049 Moss Rd.,Had trash at road last night but still missed pickup this morning,NaN,True


**Result:** The rule works as intended. These are four rows with **consecutive request numbers (28848 to 28851)**, the same date and an identical description. That is one phone call entered four times. The rule keeps the first row and marks the other three as `same_day_repeat`. Without it, this address would generate **$600 in fines for a single missed pickup**.

The hauler is also blank here, and no other request at this address names one, so this pickup can't be credited to Red River. It's one of the **245 missed pickups that still have no hauler** after the fill step.

**Check 4: The address with the most missed pickups, counted by hand.**

In [14]:
# [CHECK] 4: the busiest address after cleaning. Count its rows by hand from the raw data.
top_address = missed['clean_address'].value_counts().idxmax()
print('Busiest address:', top_address, '| missed pickups counted by code:',
      missed['clean_address'].value_counts().max())

# Every raw row for this address, all request types, so I can count the missed pickups myself
raw_rows = trash.loc[trash['clean_address'] == top_address, cols].sort_values('Date Opened')
print('All requests at this address, by type:')
print(raw_rows['Request'].value_counts().to_string())
print('Distinct raw spellings:', raw_rows['Incident Address'].nunique())
print('Haulers on its missed pickups:', missed.loc[missed['clean_address'] == top_address, 'hauler'].value_counts().to_dict())
print("Red River's busiest addresses:",
      missed.loc[missed['hauler'] == 'RED RIVER', 'clean_address'].value_counts().head(3).to_dict())
raw_rows

Busiest address: 110 GEORGE L DAVIS BLVD | missed pickups counted by code: 25
All requests at this address, by type:
Request
Trash - Curbside/Alley Missed Pickup    25
Trash Collection Complaint               4
Distinct raw spellings: 5
Haulers on its missed pickups: {'METRO': 25}


Red River's busiest addresses: {'12546 OLD HICKORY BLVD': 21, '1584 BELL RD': 19, '5135 HICKORY HOLLOW PKWY': 18}


,Request Number,Date Opened,Request,Incident Address,Description,hauler
989,37104,2018-01-05,Trash Collection Complaint,110 GEORGE L DAVIS BLVD,missed pick up again & again,METRO
2335,53873,2018-03-15,Trash Collection Complaint,110 George L Davis Blvd,missed all the time - said no alley access,METRO
2496,55760,2018-03-23,Trash - Curbside/Alley Missed Pickup,110 George L Davis Blvd,missed all the time - said no alley access {carts are placed where they always been for 6 years. Has to ke...,METRO
3793,71125,2018-05-30,Trash - Curbside/Alley Missed Pickup,110 George L Davis Blvd,trash is full needs pick up. Matt stated that the pickup has been all days of the week and never on the co...,METRO
3977,73434,2018-06-07,Trash - Curbside/Alley Missed Pickup,110 George L Davis Blvd,Missed. Caller states that this is a business and they have been missed every week. Caller wants a call ba...,METRO
4260,77172,2018-06-21,Trash - Curbside/Alley Missed Pickup,110 George L Davis Blvd,trash is full needs pick up. Matt stated that the pickup has been all days of the week and never on the co...,METRO
4798,82908,2018-07-13,Trash - Curbside/Alley Missed Pickup,110 George L Davis Blvd,Missed. Caller states that this is a business and they have been missed every week. Thursday's pickup. {Cu...,METRO
5853,94313,2018-08-31,Trash - Curbside/Alley Missed Pickup,110 George L Davis Blvd,Missed. Caller states that this is a business and they have been missed every week. Thursday's pickup. {Cu...,METRO
7787,122089,2018-12-07,Trash - Curbside/Alley Missed Pickup,110 George L Davis Blvd,missed carts,METRO
7813,122323,2018-12-10,Trash Collection Complaint,110 George L Davis Blvd,Missed again- he has to call about every week. Wanted to speak with supervisor. Transferred to Josh.,METRO


**Hand count:** I counted the `Trash - Curbside/Alley Missed Pickup` rows in the table above and got **25**, which matches the code. The 4 complaint rows are correctly left out. This address teaches two lessons:

* **Cleaning mattered.** The 25 missed pickups were spread across **5 different spellings**, and the most common single spelling had only 12. Without standardizing, this address would look about half as bad as it is.
* **The busiest address is Metro's, not Red River's.** All 25 rows have hauler `METRO`, so none of them should become a Red River fine. If I had skipped the hauler filter, the address with the most fines in my report would be one Red River isn't responsible for. Red River's busiest addresses are listed in the output above.

One related row, `110 george l davis blvd E`, stays separate because a trailing letter is treated as a unit (Part 1 assumption).

**Check 5: Nothing that isn't a missed pickup got in.**

In [15]:
# [CHECK] 5: only the two missed-pickup request types, no blank addresses, no same-day duplicates
print('Request types in result:', missed['Request'].unique().tolist())
print('Blank clean addresses:  ', missed['clean_address'].isna().sum())
print('Same-day duplicates:    ', missed.duplicated(['clean_address', 'Date Opened']).sum())
print('Damage rows included:   ', (missed['Request'] == 'Damage to Property').sum())

Request types in result: ['Trash - Curbside/Alley Missed Pickup', 'Trash - Backdoor']
Blank clean addresses:   0
Same-day duplicates:     0
Damage rows included:    0


---
## Part 2 summary
Part 2 turns **20,226 raw service requests** into **17,243 missed pickups at 11,133 standardized addresses**. Of those, **12,946 missed pickups at 8,246 addresses belong to Red River**.

| Step | What it does | Evidence it works |
|---|---|---|
| Address cleaning | 14,121 raw spellings become 12,341 addresses | 1,493 addresses combined from 2+ spellings; no city or ZIP left over; West End Ave untouched |
| Missed-pickup types | Curbside/Alley + Backdoor only | 10-row sample all real misses; complaints checked and left out on purpose |
| Usable address | Drops 48 missed-pickup rows with no house number | Intersections and bare street names can't be one premises |
| One per address per day | Removes 366 repeat reports | 4049 Moss Rd: 4 entries of one call become 1 |
| Hauler | `Metro` = `METRO`; 542 blanks filled from the same address | The busiest address (25 misses) is Metro's and won't be fined |

**Limitations carried into Part 3:**

* 245 missed pickups still have no hauler and can't be credited to Red River.
* Whole-street misses are under-reported.
* Complaints that may describe extra misses are left out.

All three push the totals **down**, so the fines in Part 3 are a floor, not a ceiling.

**Next (Part 3):** number each Red River address's missed pickups in date order with `groupby().cumcount()`. The first is $0 and every later one is $200.